In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py

In [2]:
# ASTRID cosmology values
hubble, z = 0.6774, 2.5

In [3]:
data_on = '/pscratch/sd/l/lflores/ASTRID/data/spectra_ASTRID_z2.5_500x500x2500.hdf5'  # With self-shielding

with h5py.File(data_on, 'r') as f:
    print('Keys:', f.keys())
    header_on = f['Header']
    print('----- Header -----')
    for attr in header_on.attrs:           print(f"{attr} : {header_on.attrs[attr]}")
    print('----- Data -----')
    colden = f['colden/H/1'][:]
    print('colden shape:', colden.shape)

Keys: <KeysViewHDF5 ['Header', 'colden', 'spectra', 'tau']>
----- Header -----
Hz : 252.87249366801257
box : 250000.0
discarded : 0
hubble : 0.6774
nbins : 2500
npart : [165988309584 166375000000            0            0  10297622146
     11325619]
omegab : 0.0486
omegal : 0.6911
omegam : 0.3089
redshift : 2.499999947500001
----- Data -----
colden shape: (250000, 2500)


In [4]:
print('----- Useful information -----')
Lbox = 250  # Mpc/h
print('box size:', Lbox, 'Mpc/h')

# Number of skewers per side
Nsk = 500  # colden.on_shape[0] gives the size of the axis
print(Nsk,'skewers per side')

# Number of pixels per skewer
Np = 2500 # colden.on_shape[1] gives the size of the columns
print(Np, 'pixels per skewer')

# Pixel width 
Pw = Lbox/Np  # Mpc/h 
print(Pw, 'Mpc/h pixel width')
# We are dividing the total box width in comoving units by the number of pixels in each skewer

# Minimum separation between skewers
Ssk = Lbox/Nsk  # Mpc/h 
print(Ssk, 'Mpc/h skewer separation')
# We are dividing the total box width in comoving units by the number of skewers per side

----- Useful information -----
box size: 250 Mpc/h
500 skewers per side
2500 pixels per skewer
0.1 Mpc/h pixel width
0.5 Mpc/h skewer separation


Let's use only one minibox and once I know the code is owrking, I can create a script that iterates the cod

In [4]:
data = '/pscratch/sd/l/lflores/astrid_hcd_outputs/ffts_0.30_25'

with h5py.File(data, 'r') as f:
    print('Atributes:')
    for m in f.attrs.keys():
        print(f'{m} = {f.attrs[m]}')
    print('----------------') 
    print('Data:')
    print(f.keys())

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/pscratch/sd/l/lflores/astrid_hcd_outputs/ffts_0.30_25', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

# Radial separation

In [11]:
ix, iy = np.divmod(np.arange(nskw), mb_size)  # This gives me the coordinates of each skewer within the minibox
dx = ix[:, None] - ix[None, :]
dy = iy[:, None] - iy[None, :]
dr = np.sqrt(dx**2 + dy**2)*Ssk*hubble  # This matrix contains the radial distance (Mpc) between all skewers within the minibox

In [12]:
r_bins = np.linspace(dr.min(), dr.max(), 6)  # Mpc - 5 radial bins
print('Radial edges (Mpc) to the', len(r_bins), 'defined radial bins:', r_bins)

Radial edges (Mpc) to the 6 defined radial bins: [ 0.          6.70660211 13.41320422 20.11980634 26.82640845 33.53301056]


# One minibox

## Nhi mask

In [13]:
logNHi_min, logNHi_max = 0, 21.3

In [14]:
nhi_valid = (colden_mb[0].min(axis=1) >= 10**logNHi_min) & (colden_mb[0].max(axis=1) <= 10**logNHi_max)
nhi_mask = nhi_valid[:, None] & nhi_valid[None, :]
print(nhi_mask.shape)

(5000, 5000)


In [15]:
fft_tot_mb[0].shape

(5000, 1251)

In [16]:
r_min, r_max = r_bins[0], r_bins[1]
r_mask = (dr >= r_min) & (dr < r_max)
mask = r_mask & nhi_mask

In [17]:
def Px_calc(f1, f2, mask, Lbox, Np):
    """ This function calculates Px between two fields f1 and f2, masking those skewers that don't fulfill the mask condition.
    
    Parameters:
    ----------------------
    f1 : n-array
        FFTs of the field 1 for all the skewers
    d2 : n-array
        FFTs of the field 2 for all the skewers
    mask : n-array with boleean values
        Mask to select skewers
    Lbox: value
        Box size (Will determine the units of Px)
    Np : value
        Numbe of pixels per skewer

    Returns:
    ---------------------
    Px: n-array

    """

    i_idx, j_idx = np.where(np.triu(mask, k=1))  # Coordinates of selected skewers
    
    A = f1[i_idx]  # Field A
    B = f2[j_idx].conj()  # Field 2
    Px = np.sum(A*B, axis=0)
    
    npairs = len(i_idx)  # Number of pairs considered
    if npairs > 0:
        Px *= Lbox/(npairs*(Np**2))  # Normalization
    else:
        Px[:] = 0
        
    return Px, npairs   


In [1]:
fft_tot_mb[0].shape

NameError: name 'fft_tot_mb' is not defined

In [ ]:
Px_tot, _ = Px_calc(fft_tot_mb[0], fft_tot_mb[0], mask, Lbox=Lbox*hubble, Np=Np)

In [20]:
mask.shape

(5000, 5000)

In [21]:
i_idx, j_idx = np.where(np.triu(mask, k=1))  # Coordinates of selected skewers

In [2]:
i_idx.shape

NameError: name 'i_idx' is not defined

In [ ]:
field1 = fft_tot_mb[0, i_idx, :]

In [14]:
for r_value in np.arange(len(r_bins)-1):
    r_min, r_max = r_bins[r_value], r_bins[r_value+1]
    r_mask = (dr >= r_min) & (dr < r_max)
    mask = r_mask & nhi_mask

    Px = np.zeros_like(fft_tot_mb[0])
    npairs = 0
    for i in range(nskw):
        for j in range(i+1, nskw):
            if mask[i, j]:
                field1 = fft_tot_mb[0, i, :]
                field2 = fft_tot_mb[0, j, :]
                Px += field1*field2.conjugate()
                npairs += 1
                
    if npairs > 0:
        Px *= Lbox/(npairs*(Np**2))
    else:
        Px[:] = 0.0

    

KeyboardInterrupt: 

In [18]:
def Px_calc(f1, f2, mask, Lbox, Np):
    """ This function calculates Px between two fields f1 and f2, masking those skewers that don't fulfill the mask condition.
    
    Parameters:
    ----------------------
    f1 : n-array
        FFTs of the field 1 for all the skewers
    d2 : n-array
        FFTs of the field 2 for all the skewers
    mask : n-array with boleean values
        Mask to select skewers
    Lbox: value
        Box size (Will determine the units of Px)
    Np : value
        Numbe of pixels per skewer

    Returns:
    ---------------------
    Px: n-array

    """

    i_idx, j_idx = np.where(np.triu(mask, k=1))  # Coordinates of selected skewers
    
    A = f1[i_idx]  # Field A
    B = f2[j_idx].conj()  # Field 2
    Px = np.sum(A*B, axis=0)
    
    npairs = len(i_idx)  # Number of pairs considered
    if npairs > 0:
        Px *= Lbox/(npairs*(Np**2))  # Normalization
    else:
        Px[:] = 0
        
    return Px, npairs   
